## Building Agents
In previous notebooks we talked about a number of workflows that used LLMs in some kind of reasoning scaffolding. Patterns such as _Routing_, _Orchestrator-Worker_, _Evaluator-Optimizer_, you let the LLM decide how the control flows through the scaffolding. Now let's remove the scaffolding and talk about **Agents**.

With **Agents** you are simply allowing the LLM to perform actions, in the form of tool calls, and directly receive the output or feedback from those actions. So in the workflow cases we talked about there are pre-defined code paths that we had the LLM follow, but in the case of an Agent we remove those.

When do you actually need an Agent? You see Agents being used in cases where you have open ended problems that you cannot easily capture in a workflow. For example, you want an LLM to utilize different tools in a patterns that you just cannot predict. Agents are not guaranteed to execute tasks in a certain sequence. If your process warrents execution of tasks in sequence, prefer workflows - one of the patterns we discussed previously.

<div align="center">
<img src="images/07-Agent.png" width="450" heigh="250" alt="Orchestrator Workflow"/>
</div>

In this example we'll build an Agent with tools.

In [1]:
from dotenv import load_dotenv
from time import sleep
from rich.console import Console
from rich.markdown import Markdown
from typing import TypedDict, Annotated, List
import operator
from pydantic import BaseModel, Field

from langchain.chat_models import init_chat_model
from langchain_core.messages import SystemMessage, HumanMessage

from langgraph.graph import StateGraph, START, END
from langgraph.types import Send

In [2]:
load_dotenv(override=True)

console = Console()

In [3]:
# create our LLM - we'll be using Google Gemini flash - let;s make it creative
llm = init_chat_model("google_genai:gemini-2.0-flash", temperature=0.4)

c:\Users\BHOBEMRMANISHJAGDISH\Dev\code\git_projects\learning_langgraph\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [6]:
# let's build some math tools
from langchain_core.tools import tool


@tool
def multiply(a: int, b: int) -> int:
    """multiplies 2 integers a & b and returns product

    Args:
        a - first int
        b - second int
    """
    return a * b


@tool
def add(a: int, b: int) -> int:
    """adds 2 integers a & b and returns sum

    Args:
        a - first int
        b - second int
    """
    return a + b


@tool
def divide(a: int, b: int) -> float:
    """Divide `a` and `b`.

    Args:
        a: First int
        b: Second int
    """
    return a / b


@tool
def subtract(a: int, b: int) -> float:
    """subtracts b from a and returns the difference

    Args:
        a: First int
        b: Second int
    """
    return a - b


# Augment the LLM with tools
tools = [add, multiply, divide, subtract]
tools_by_name = {tool.name: tool for tool in tools}
llm_with_tools = llm.bind_tools(tools)

In [9]:
from langgraph.graph import MessagesState
from langchain_core.messages import SystemMessage, HumanMessage, ToolMessage


# Nodes
def llm_call(state: MessagesState):
    """LLM decides whether to call a tool or not"""

    return {
        "messages": [
            llm_with_tools.invoke(
                [
                    SystemMessage(
                        content="You are a helpful assistant tasked with performing arithmetic on a set of inputs."
                    )
                ]
                + state["messages"]
            )
        ]
    }


def tool_node(state: dict):
    """Performs the tool call"""

    result = []
    for tool_call in state["messages"][-1].tool_calls:
        tool = tools_by_name[tool_call["name"]]
        observation = tool.invoke(tool_call["args"])
        result.append(ToolMessage(content=observation, tool_call_id=tool_call["id"]))
    return {"messages": result}


# Conditional edge function to route to the tool node or end based upon whether the LLM made a tool call
def should_continue(state: MessagesState) -> Literal["tool_node", END]:
    """Decide if we should continue the loop or stop based upon whether the LLM made a tool call"""

    messages = state["messages"]
    last_message = messages[-1]

    # If the LLM makes a tool call, then perform an action
    if last_message.tool_calls:
        return "tool_node"

    # Otherwise, we stop (reply to the user)
    return END

NameError: name 'Literal' is not defined

In [ ]:
# Build workflow
agent_builder = StateGraph(MessagesState)

# Add nodes
agent_builder.add_node("llm_call", llm_call)
agent_builder.add_node("tool_node", tool_node)

# Add edges to connect nodes
agent_builder.add_edge(START, "llm_call")
agent_builder.add_conditional_edges("llm_call", should_continue, ["tool_node", END])
agent_builder.add_edge("tool_node", "llm_call")

# Compile the agent
agent = agent_builder.compile()

# Show the agent
display(Image(agent.get_graph(xray=True).draw_mermaid_png()))

# Invoke
messages = [HumanMessage(content="Add 3 and 4.")]
messages = agent.invoke({"messages": messages})
for m in messages["messages"]:
    m.pretty_print()